In [1]:
# ============================================================
# Cell 1 — Install
# ============================================================
!pip install -q -U "transformers>=5.0.0" datasets accelerate scikit-learn

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.


In [2]:
# ============================================================
# Cell 2 — Imports & Config
# ============================================================
import os, re, random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from datasets import Dataset, ClassLabel
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ── Config ────────────────────────────────────────────────
# roberta-base     → best accuracy, ~500MB  (may need paid Render tier)
# distilroberta-base → great accuracy, ~330MB (borderline free tier)
# distilbert-base-uncased → current, ~268MB (safe for free tier)
MODEL_NAME  = "roberta-base"
MAX_LENGTH  = 256
BATCH_SIZE  = 16
EPOCHS      = 4
LR          = 2e-5
SEED        = 42
# ──────────────────────────────────────────────────────────

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Model  :", MODEL_NAME)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

Model  : roberta-base
GPU    : Tesla T4


In [4]:
# ============================================================
# Cell 3 — Upload files
# ============================================================
from google.colab import files
uploaded = files.upload()
# Upload: Fake.csv, True.csv, train.tsv, valid.tsv, test.tsv

Saving Fake.csv to Fake.csv
Saving test.tsv to test.tsv
Saving train.tsv to train.tsv
Saving True.csv to True.csv


In [5]:
# ============================================================
# Cell 4 — Text Cleaning  (fixes the Reuters shortcut problem)
# ============================================================
DATELINE_RE   = re.compile(r'^[A-Z][A-Z\s,\.]{2,}\s*\([^)]+\)\s*[-–]\s*')
SOURCE_TAG_RE = re.compile(r'\((Reuters|AP|AFP|UPI|CNN|BBC|NPR|REUTERS)\)',
                           re.IGNORECASE)
HTML_TAG_RE   = re.compile(r'<[^>]+>')
WHITESPACE_RE = re.compile(r'\s+')

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = DATELINE_RE.sub('', text)       # "WASHINGTON (Reuters) -"
    text = SOURCE_TAG_RE.sub('', text)     # stray "(Reuters)" in body
    text = HTML_TAG_RE.sub('', text)       # HTML artifacts
    text = WHITESPACE_RE.sub(' ', text)    # normalise whitespace
    return text.strip()

# ── Sanity check ──────────────────────────────────────────
tests = [
    "WASHINGTON (Reuters) - The Fed held rates steady.",
    "NEW YORK (AP) - Markets closed lower on Friday.",
    "The government announced a new policy. (Reuters)",
    "Normal sentence with no dateline.",
]
for t in tests:
    print(f"IN : {t}")
    print(f"OUT: {clean_text(t)}\n")

IN : WASHINGTON (Reuters) - The Fed held rates steady.
OUT: The Fed held rates steady.

IN : NEW YORK (AP) - Markets closed lower on Friday.
OUT: Markets closed lower on Friday.

IN : The government announced a new policy. (Reuters)
OUT: The government announced a new policy.

IN : Normal sentence with no dateline.
OUT: Normal sentence with no dateline.



In [6]:
# ============================================================
# Cell 5 — Load Fake / True CSVs
# ============================================================
fake_df = pd.read_csv("Fake.csv", engine="python", on_bad_lines="skip")
fake_df["label"] = 0
fake_df["text"]  = (
    fake_df["title"].fillna("") + " " + fake_df["text"].fillna("")
).apply(clean_text)
fake_df = fake_df[["text", "label"]]

real_df = pd.read_csv("True.csv", engine="python", on_bad_lines="skip")
real_df["label"] = 1
real_df["text"]  = (
    real_df["title"].fillna("") + " " + real_df["text"].fillna("")
).apply(clean_text)
real_df = real_df[["text", "label"]]

main_df = pd.concat([fake_df, real_df], ignore_index=True)
main_df = main_df[main_df["text"].str.len() > 20]
main_df = main_df.drop_duplicates(subset=["text"])
main_df["label"] = main_df["label"].astype(int)

# ── Verify no dateline leakage ────────────────────────────
reuters_left = main_df["text"].str.contains(
    r'\(Reuters\)', case=False, regex=True
).sum()
print(f"Fake/True size  : {len(main_df)}")
print(f"'(Reuters)' left: {reuters_left}  (should be 0 or near 0)")
print(main_df["label"].value_counts())

Fake/True size  : 39100
'(Reuters)' left: 0  (should be 0 or near 0)
label
1    21195
0    17905
Name: count, dtype: int64


In [9]:
# ============================================================
# Cell 6 — Load LIAR Dataset
# ============================================================
LIAR_COLS   = ["id","label","statement","subjects","speaker","job_title",
               "state","party","barely_true_count","false_count",
               "half_true_count","mostly_true_count","pants_on_fire_count","context"]
FAKE_LABELS = {"pants-fire", "false", "barely-true"}
REAL_LABELS = {"mostly-true", "true"}

def load_liar(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", header=None, names=LIAR_COLS)
    df = df[["statement", "label"]].copy()
    df = df[df["label"].isin(FAKE_LABELS | REAL_LABELS)]
    df["label"] = df["label"].apply(lambda x: 0 if x in FAKE_LABELS else 1)
    df = df.rename(columns={"statement": "text"})
    df["text"] = df["text"].apply(clean_text)
    df = df[df["text"].str.len() > 5].drop_duplicates(subset=["text"])
    return df.reset_index(drop=True)

liar_train_df = load_liar("train.tsv")
liar_test_df  = load_liar("test.tsv")

# Use train only for training, test held back for final eval
liar_for_training = liar_train_df.copy()

print(f"LIAR training pool : {len(liar_for_training)}")
print(f"LIAR test (held)   : {len(liar_test_df)}")
print(liar_for_training["label"].value_counts())


LIAR training pool : 8116
LIAR test (held)   : 1002
label
0    4480
1    3636
Name: count, dtype: int64


In [10]:
# ============================================================
# Cell 7 — Combine & Balance
# ============================================================
# 3x oversample LIAR so short political statements
# influence training alongside long articles
liar_oversampled = pd.concat([liar_for_training] * 3, ignore_index=True)

combined_df = pd.concat([main_df, liar_oversampled], ignore_index=True)
combined_df["text"]  = combined_df["text"].astype(str).str.strip()
combined_df["label"] = combined_df["label"].astype(int)
combined_df = combined_df[combined_df["text"].str.len() > 10]
combined_df = combined_df.drop_duplicates(subset=["text"])
combined_df = combined_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Combined total : {len(combined_df)}")
print(combined_df["label"].value_counts())
print(f"Class balance  : {combined_df['label'].mean():.2%} real")

Combined total : 47216
label
1    24831
0    22385
Name: count, dtype: int64
Class balance  : 52.59% real


In [11]:
# ============================================================
# Cell 8 — Train / Val / Test Split  (80 / 10 / 10)
# ============================================================
train_df, temp_df = train_test_split(
    combined_df, test_size=0.20,
    stratify=combined_df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50,
    stratify=temp_df["label"], random_state=SEED
)

for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    df.reset_index(drop=True, inplace=True)
    print(f"{name:5s}: {len(df):6d} samples  |  {df['label'].mean():.2%} real")

Train:  37772 samples  |  52.59% real
Val  :   4722 samples  |  52.60% real
Test :   4722 samples  |  52.58% real


In [12]:
# ============================================================
# Cell 9 — Tokenise
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(df[["text", "label"]], preserve_index=False)
    ds = ds.rename_column("label", "labels")
    ds = ds.cast_column("labels", ClassLabel(names=["fake", "real"]))
    return ds

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

def prepare(df: pd.DataFrame) -> Dataset:
    ds = to_hf_dataset(df)
    ds = ds.map(tokenize, batched=True, remove_columns=["text"])
    if "token_type_ids" in ds.column_names:
        ds = ds.remove_columns(["token_type_ids"])
    ds.set_format(type="torch")
    return ds

train_dataset = prepare(train_df)
val_dataset   = prepare(val_df)
test_dataset  = prepare(test_df)

print("Columns :", train_dataset.column_names)
print("Shape   :", train_dataset[0]["input_ids"].shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Casting the dataset:   0%|          | 0/4722 [00:00<?, ? examples/s]

Map:   0%|          | 0/4722 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/4722 [00:00<?, ? examples/s]

Map:   0%|          | 0/4722 [00:00<?, ? examples/s]

Columns : ['labels', 'input_ids', 'attention_mask']
Shape   : torch.Size([256])


In [13]:
# ============================================================
# Cell 10 — Load Model
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    problem_type="single_label_classification"
)

params = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL_NAME}  |  Parameters: {params:,}")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: roberta-base  |  Parameters: 124,647,170


In [14]:
# ============================================================
# Cell 11 — Metrics
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    return {
        "accuracy" : round(acc, 4),
        "f1"       : round(f1, 4),
        "precision": round(precision, 4),
        "recall"   : round(recall, 4),
    }

In [15]:
# ============================================================
# Cell 12 — Training Arguments
# ============================================================
training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,        # effective batch = 32
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.1,
    max_grad_norm=1.0,                    # gradient clipping

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),

    logging_steps=100,
    report_to="none",
    seed=SEED,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [16]:
# ============================================================
# Cell 13 — Train
# ============================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.243266,0.114439,0.940100,0.942300,0.954600,0.930400
2,0.208977,0.118533,0.940700,0.944500,0.929800,0.959700
3,0.178440,0.163383,0.936300,0.940500,0.924500,0.956900
4,0.128386,0.232311,0.933100,0.937400,0.923400,0.951700


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4724, training_loss=0.2255494504233319, metrics={'train_runtime': 2046.349, 'train_samples_per_second': 73.833, 'train_steps_per_second': 2.309, 'total_flos': 1.987646156611584e+16, 'train_loss': 0.2255494504233319, 'epoch': 4.0})

In [17]:
# ============================================================
# Cell 14 — Full Evaluation Report
# ============================================================
from torch.utils.data import DataLoader

def full_eval(model, dataset, name="Dataset"):
    model.eval()
    device = next(model.parameters()).device
    loader = DataLoader(dataset, batch_size=32)
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids    = batch["input_ids"].to(device)
            mask   = batch["attention_mask"].to(device)
            lbls   = batch["labels"].to(device)
            logits = model(input_ids=ids, attention_mask=mask).logits
            preds  = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(classification_report(
        all_labels, all_preds,
        target_names=["FAKE", "REAL"], digits=4
    ))
    cm = confusion_matrix(all_labels, all_preds)
    print("Confusion Matrix:")
    print(pd.DataFrame(
        cm,
        index=["Actual FAKE", "Actual REAL"],
        columns=["Pred FAKE",  "Pred REAL"]
    ))

# Prepare LIAR test set
liar_test_dataset = prepare(liar_test_df)

torch.cuda.empty_cache()
full_eval(model, val_dataset,       "Validation Set")
full_eval(model, test_dataset,      "Held-out Test Set")
full_eval(model, liar_test_dataset, "LIAR Test Set (out-of-distribution)")

Casting the dataset:   0%|          | 0/1002 [00:00<?, ? examples/s]

Map:   0%|          | 0/1002 [00:00<?, ? examples/s]


  Validation Set
              precision    recall  f1-score   support

        FAKE     0.9537    0.9196    0.9363      2238
        REAL     0.9298    0.9597    0.9445      2484

    accuracy                         0.9407      4722
   macro avg     0.9417    0.9397    0.9404      4722
weighted avg     0.9411    0.9407    0.9406      4722

Confusion Matrix:
             Pred FAKE  Pred REAL
Actual FAKE       2058        180
Actual REAL        100       2384

  Held-out Test Set
              precision    recall  f1-score   support

        FAKE     0.9502    0.9205    0.9351      2239
        REAL     0.9303    0.9565    0.9432      2483

    accuracy                         0.9394      4722
   macro avg     0.9402    0.9385    0.9392      4722
weighted avg     0.9397    0.9394    0.9394      4722

Confusion Matrix:
             Pred FAKE  Pred REAL
Actual FAKE       2061        178
Actual REAL        108       2375

  LIAR Test Set (out-of-distribution)
              precision    r

In [18]:
# ============================================================
# Cell 15 — Dateline Bias Test
#   If the fix worked, WITH and WITHOUT dateline should score
#   similarly.  Big gap = shortcut still present.
# ============================================================
import torch.nn.functional as F
LABEL_MAP = {0: "FAKE", 1: "REAL"}

def predict(text: str) -> dict:
    model.eval()
    device = next(model.parameters()).device
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, padding="max_length", max_length=MAX_LENGTH
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        probs = F.softmax(model(**inputs).logits, dim=-1)[0].cpu().numpy()
    pred_id = int(np.argmax(probs))
    return {
        "label"     : LABEL_MAP[pred_id],
        "prob_fake" : round(float(probs[0]), 3),
        "prob_real" : round(float(probs[1]), 3),
    }

examples = {
    "✅ With Reuters dateline" :
        "WASHINGTON (Reuters) - The Federal Reserve held rates steady on Wednesday.",
    "✅ Without dateline" :
        "The Federal Reserve held rates steady on Wednesday.",
    "❌ Obvious fake" :
        "The government is secretly poisoning the water supply to control the population.",
    "⚠️  Borderline" :
        "A senior official confirmed a proposal to limit social media for users under 21.",
    "❌ Classic fake style" :
        "Hillary Clinton was caught on tape admitting she rigged the primary against Bernie Sanders.",
}

print(f"\n{'='*70}")
print("  BIAS TEST — with/without dateline should score similarly")
print(f"{'='*70}")
for label, text in examples.items():
    r = predict(text)
    bar = "█" * int(r["prob_real"] * 20)
    print(f"\n{label}")
    print(f"  → {r['label']}  fake:{r['prob_fake']:.3f}  real:{r['prob_real']:.3f}  {bar}")


  BIAS TEST — with/without dateline should score similarly

✅ With Reuters dateline
  → REAL  fake:0.000  real:1.000  ████████████████████

✅ Without dateline
  → REAL  fake:0.017  real:0.983  ███████████████████

❌ Obvious fake
  → FAKE  fake:0.875  real:0.125  ██

⚠️  Borderline
  → FAKE  fake:0.766  real:0.234  ████

❌ Classic fake style
  → FAKE  fake:0.965  real:0.035  


In [19]:
# ============================================================
# Cell 16 — Save Model
# ============================================================
SAVE_DIR = "./news_credibility_model_v2"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to: {SAVE_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: ./news_credibility_model_v2


In [21]:
# ============================================================
# Cell 17 — Push to HuggingFace
# ============================================================
from huggingface_hub import login
login()

model.push_to_hub("V1gnesh/fake-news-model")
tokenizer.push_to_hub("V1gnesh/fake-news-model")
print("Pushed!")

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...1qio7c4/model.safetensors:   0%|          |  551kB /  499MB            

Pushed!


In [22]:
# ============================================================
# Cell 18 — Download zip
# ============================================================
import shutil
from google.colab import files

shutil.make_archive("news_credibility_model_v2", "zip", ".", SAVE_DIR)
files.download("news_credibility_model_v2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>